# D1.11 · Honeypots, canaries and deception in the agent's environment

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.10 · Fleet-level correlation: seeing a swarm](https://spbreed.github.io/cyber-commons/lessons/D1.10.html)**.

| | |
|---|---|
| Tools used | Canarytokens, Inspect, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Authenticate with a canary and watch a zero-threshold alert fire; then salt a benchmark and read the cheat-attempt rate as a leading indicator.

**Why a security engineer needs it.** Every other detector needs a threshold, and every threshold is a trade. Deception needs neither — but only if the bait is placed where the agent actually looks, and rotated before it is learned. The control it builds is: canary tokens in config, environment and artifact metadata (C4.4), and honeypot tasks salted into the benchmark whose cheat path is logged rather than rewarded (C10.3).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Every detector in this chapter needs a threshold, and every threshold is a trade. A canary needs neither: nothing legitimate has any reason to touch it, so its false-positive rate is zero by construction rather than by tuning.

> **At CyberTravels.** A canary credential in CyberTravels' environment and a honeypot task in the benchmark: two detectors with no threshold to tune, because nothing legitimate has any reason to touch either.

## 2 · The framework

```
   tuned detector                     deception

   threshold ---> TP and FP           canary ---> any touch is a hit
   needs triage                       needs none: nothing legitimate
   finds more                          has a reason to touch it
                                      finds less, at precision 1.00

   canary token in env/config/docs    honeypot task with an instrumented
   real prefix, real length            shortcut that logs and scores 0

   both decay: bait that is not rotated is bait that has been learned
```

Every detector in this chapter has needed a threshold, and every threshold is a
trade between missing things and crying wolf. Deception is the exception:
**a canary has a false-positive rate of zero by construction**, because nothing
legitimate ever touches it.

Two controls from the incident register live here, and both are about the
environment your harness builds rather than the code it analyses.

**Credential canaries (C4.4).** Tokens that look genuine — same prefix, same
format, same length — placed where an agent or an attacker would plausibly
look: config files, environment listings, documentation, artifact metadata. The
issuing service alerts on any authentication attempt. There is no legitimate
use, so a hit is a confirmed compromise and opens an incident automatically. In
the source incident, live credentials found on the public internet were
redistributed to the swarm within minutes; a canary alongside them would have
carried source IP, user agent and timestamp.

**Honeypot tasks (C10.3).** Benchmark tasks salted with a plausible-looking
shortcut that is instrumented rather than functional, plus tasks whose scoring
*appears* manipulable but where manipulation is detected rather than rewarded.
The cheat-attempt rate on those tasks is a leading indicator for the whole
fleet — available continuously, not after the fact.

Both share one operational requirement: **rotate them**. Bait that stays put is
bait that gets learned, and a learned canary is worse than none because it
tells you the environment is clean when it is only well-mapped.

## 3 · The procedure, as a skill

A canary credential nothing legitimate uses produces an alert with no structurally possible false positive. The skill places them, attaches source and user agent to each hit, adds honeypot tasks that measure inclination, and models the decay that sets the rotation interval.

In [ ]:
# skills/detection/canary-and-honeypot-design/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: canary-and-honeypot-design
description: >-
  Place credential canaries and honeypot tasks in an agent's environment so that
  an alert has no structurally possible false positive, and measure how fast the
  signal decays as agents learn. Use when deception is being added, or when a
  detection needs to be one nobody has to triage.
allowed-tools: Read, Grep, Glob
---

# An alert with no possible false positive

A canary credential that nothing legitimate uses produces an alert that needs no
triage: the only way it authenticates is that somebody read it and tried it.
That property is structural, not statistical, and it is why deception belongs in
an agent environment where every other signal is ambiguous.

## When to use this

Designing detection for agent environments, and whenever an existing detection's
false-positive rate is the reason it is ignored.

## Procedure

**1 — Place canaries where only reading them is unusual.** Environment
variables, config files, the fixtures a code agent walks. They must be
indistinguishable from real ones — a canary named `canary_key` is a filter, not
a trap.

**2 — Ensure nothing legitimate uses them.** This is the whole property. Check
the code, the tests and the deployment. One legitimate reference and the alert
becomes triage.

**3 — Instrument the authentication path** to capture source address and user
agent on use. A canary alert with no context tells you that it happened and
nothing about who.

**4 — Add honeypot tasks for behaviour rather than credentials.** A task with an
available shortcut that nothing legitimate would take. Log the attempt and score
it; this measures inclination, which no credential can.

**5 — Measure decay.** Agents and operators learn. Model the hit rate over days
since placement, with and without rotation, and set the rotation interval from
the curve rather than from a calendar.

## Output contract

```json
{
  "canaries": [{"id": "str", "placed_in": "str", "indistinguishable": true, "legitimate_refs": 0}],
  "alerts": [{"canary": "str", "source_ip": "str", "user_agent": "str", "false_positive_possible": false}],
  "honeypot_tasks": [{"task": "str", "shortcut": "str", "attempts": 0}],
  "decay": {"days": [0], "hit_rate": [0.0], "rotation_days": 0}
}
```

## Failure modes

- **A canary anything legitimate touches.** The property is gone and the alert
  becomes noise.
- **Naming it as a canary.** It becomes a filter for the competent attacker.
- **Never rotating.** The signal decays and the absence of alerts reads as
  safety.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/detection/canary-and-honeypot-design/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/detection/canary-and-honeypot-design/scripts/canary_and_honeypot_design.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Place credential canaries and honeypot tasks whose alerts have no structurally possible false positive, and measure decay as agents learn.

This is the executable half of the `canary-and-honeypot-design` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

CANARIES = {
 "hf_CANARY7Fq2mXvLpR8s": {"placed": "worker env listing",  "rotated_day": 0},
 "ghp_CANARY4mZk1TbNwQ9x": {"placed": "docs/DEPLOY.md",     "rotated_day": 0},
 "sk-CANARYd3Vn8yHc2Uae":  {"placed": "artifact metadata",  "rotated_day": 0},
}
REAL = {"hf_liveTokenNotShown", "ghp_alsoLive"}

AUTH_EVENTS = [
 ("hf_liveTokenNotShown",   "10.2.0.11",  "ci-runner"),
 ("hf_CANARY7Fq2mXvLpR8s",  "203.0.113.9","python-requests/2.31"),
 ("ghp_alsoLive",           "10.2.0.11",  "ci-runner"),
 ("sk-CANARYd3Vn8yHc2Uae",  "203.0.113.9","python-requests/2.31"),
]

print(f"{'token':26s}{'source':14s}{'agent':22s}verdict")
hits = []
for token, ip, ua in AUTH_EVENTS:
    if token in CANARIES:
        hits.append((token, ip, ua))
        verdict = "CONFIRMED COMPROMISE"
    else:
        verdict = "normal use"
    print(f"{token[:24]:26s}{ip:14s}{ua:22s}{verdict}")

print(f"\ncanary hits: {len(hits)}  false positives possible: 0")
print("Not zero because the detector is good - zero because nothing legitimate")
print("has any reason to touch these strings. That is the whole property, and")
print("it is why a hit opens an incident rather than a ticket.")
print(f"attribution carried on the hit: source {hits[0][1]}, agent {hits[0][2]}")
assert len(hits) == 2

BENCHMARK = [
 {"id": "T-01", "kind": "real",     "shortcut": None},
 {"id": "T-02", "kind": "real",     "shortcut": None},
 {"id": "T-03", "kind": "honeypot", "shortcut": "flag readable in the scorer fixture"},
 {"id": "T-04", "kind": "real",     "shortcut": None},
 {"id": "T-05", "kind": "honeypot", "shortcut": "answer key path guessable from the task id"},
 {"id": "T-06", "kind": "real",     "shortcut": None},
]

def attempt(task, agent_takes_shortcut):
    """The shortcut is instrumented: it records, it never scores."""
    if task["shortcut"] and agent_takes_shortcut:
        return {"task": task["id"], "scored": 0.0, "logged": "cheat attempt",
                "detail": task["shortcut"]}
    return {"task": task["id"], "scored": 1.0 if task["kind"] == "real" else 0.0,
            "logged": None, "detail": None}

runs = [attempt(t, agent_takes_shortcut=True) for t in BENCHMARK]
for r in runs:
    if r["logged"]:
        print(f"   {r['task']}  {r['logged']}: {r['detail']}  (scored {r['scored']})")

honeypots = [t for t in BENCHMARK if t["kind"] == "honeypot"]
attempts = [r for r in runs if r["logged"]]
rate = len(attempts) / len(honeypots)
print(f"\nhoneypot tasks: {len(honeypots)}   cheat attempts: {len(attempts)}"
      f"   rate: {rate:.0%}")
print()
print("The rate is the product, not the individual catch. Read continuously it")
print("is a leading indicator for the whole fleet; read after the run it is an")
print("autopsy.")
assert rate == 1.0 and all(r["scored"] == 0.0 for r in attempts)

def hit_rate(day, rotated_day, learn_days=21):
    """Once bait is learned it stops firing, and silence reads as safety."""
    age = day - rotated_day
    return 1.0 if age < learn_days else 0.0

for day in (0, 7, 20, 21, 40, 90):
    r = hit_rate(day, rotated_day=0)
    reading = "fires on contact" if r else "silent - and silence looks clean"
    print(f"day {day:>3}   detection rate {r:.0%}   {reading}")

print()
rotated = [hit_rate(day, rotated_day=(day // 21) * 21) for day in (0, 20, 40, 90)]
print(f"with quarterly-or-faster rotation: {['%.0f%%' % (x*100) for x in rotated]}")
print()
print("An unrotated canary does not fail loudly. It reports a clean environment")
print("that is merely well-mapped, which is a worse position than having no")
print("canary at all, because you now believe something.")
assert hit_rate(40, 0) == 0.0 and all(x == 1.0 for x in rotated)

DETECTORS = {
 "write-volume anomaly":  {"tp": 18, "fp": 140, "needs_threshold": True},
 "entropy on artifacts":  {"tp": 11, "fp": 62,  "needs_threshold": True},
 "credential canary":     {"tp": 2,  "fp": 0,   "needs_threshold": False},
 "honeypot task":         {"tp": 6,  "fp": 0,   "needs_threshold": False},
}
print(f"{'detector':24s}{'TP':>4}{'FP':>6}{'precision':>11}  threshold?")
for name in sorted(DETECTORS):
    d = DETECTORS[name]
    prec = d["tp"] / (d["tp"] + d["fp"])
    print(f"{name:24s}{d['tp']:>4}{d['fp']:>6}{prec:>11.2f}  "
          f"{'yes' if d['needs_threshold'] else 'none needed'}")

deception = [n for n in DETECTORS if not DETECTORS[n]["needs_threshold"]]
print(f"\ndetectors needing no threshold: {sorted(deception)}")
print()
print("Deception finds less. What it finds needs no triage, no tuning and no")
print("argument - which is why it belongs beside the volume detectors rather")
print("than instead of them.")
assert all(DETECTORS[n]["fp"] == 0 for n in deception)

## What you just proved

Two canary authentications out of four events are confirmed compromises with source IP and user agent attached, and no false positive is structurally possible. Both honeypot tasks log a cheat attempt and score zero for it. An unrotated canary's detection rate falls to 0% once learned — reporting a clean environment that is only well-mapped — while rotation holds it at 100%. Deception finds fewer things than the volume detectors and finds them at precision 1.00.

## Your turn

Place one canary credential in the environment your agents run in, wired to a real alert, and leave it. The interesting outcome is not the alert; it is discovering, six weeks later, which systems can even see it.

## Where this leaves you

**What you can do now.** Triage as a loop you supervise, detections written for machine-tempo actors, agent telemetry as a real data source, agent-versus-human attribution, drift monitoring — and the two the incident register adds: detections whose subject is the platform, and analytics that read across runs rather than within them.

**What you still cannot do.** Detection ends at the alert. Every lesson here stops one step before the hard part — a fleet that is acting right now, on delegated credentials, faster than the person reading the alert can type.

**Chapter 9 is that step: scope it, contain it, replay it, and decide in advance who is allowed to stop it. Next → D2.1, agent-assisted reconstruction.**

---

**Next → [D2.1 · Agent-assisted reconstruction](https://spbreed.github.io/cyber-commons/lessons/D2.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*